In [4]:
import pandas
import requests
from bs4 import BeautifulSoup

print("All libraries work!")

All libraries work!


In [5]:
url = "https://books.toscrape.com/"

response = requests.get(url)

print(response)

<Response [200]>


In [34]:
book_data = []

soup = BeautifulSoup(response.text, "html.parser")


for book in soup.find_all("article", class_="product_pod"):
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    book_data.append({"title": title, "price": price})

rating = book.find("p", class_="star-rating")
rating_value = rating.get("class")[1]
print(book.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="1000-places-to-see-before-you-die_1/index.html">
   <img alt="1,000 Places to See Before You Die" class="thumbnail" src="../media/cache/d7/0f/d70f7edd92705c45a82118c3ff6c299d.jpg"/>
  </a>
 </div>
 <p class="star-rating Five">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="1000-places-to-see-before-you-die_1/index.html" title="1,000 Places to See Before You Die">
   1,000 Places to See ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£26.08
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [36]:
all_books = []

ratings_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")
    for book in books:
        link = book.h3.a.get("href")
        image = book.find("img").get("src")
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text
        price = float(price.replace("Â£", ""))
        rating = book.find("p", class_="star-rating")
        rating_value = rating.get("class")[1]
        rating_value = ratings_map[rating_value]
        availability = book.find("p", class_="instock availability").text.strip()
        all_books.append({"Title": title, "Price": price, "Rating": rating_value, "Availability": availability, "Link": link, "Image": image})


df = pandas.DataFrame(all_books)
df.sort_values("Price", ascending=False, inplace=True)
df.sort_values("Rating", ascending=False, inplace=True)
print(df)

                                                 Title  Price  Rating  \
379  How to Speak Golf: An Illustrated Guide to Lin...  58.32       5   
560                     The Barefoot Contessa Cookbook  59.92       5   
521  Naturally Lean: 125 Nourishing Gluten-Free, Pl...  11.38       5   
601                                The Darkest Corners  11.33       5   
316                                 Dear Mr. Knightley  11.21       5   
..                                                 ...    ...     ...   
704  Unstuffed: Decluttering Your Home, Mind, and Soul  58.09       1   
805  Miracles from Heaven: A Little Girl, Her Journ...  57.83       1   
152  The Long Shadow of Small Ghosts: Murder and Me...  10.97       1   
725  The Restaurant at the End of the Universe (Hit...  10.92       1   
133  Thomas Jefferson and the Tripoli Pirates: The ...  59.64       1   

    Availability                                               Link  \
379     In stock  how-to-speak-golf-an-illustrated-g

In [37]:
len(df)

1000

In [39]:
df.to_excel("books.xlsx", index=False)